In [1]:
from pyspark.sql import SparkSession

In [25]:
from pyspark.sql.functions import col, when, sum, lit, trim, regexp_replace, lower, coalesce


In [5]:
spark = SparkSession.builder.appName("TratamentoDeDados").getOrCreate()

In [7]:
dados_ficticios = [
    ("  Ana Silva  ", "SÃO PAULO", "R$ 5.000,00", None),
    ("joao santos", "rio de janeiro", "R$ 3.200,50", 25),
    (" Maria Oliveira ", "Belo Horizonte", "R$ 12.000,00", 40),
    ("Carlos souza", None, "R$ 1.500,00", 17)
]

In [8]:
colunas = ["nome", "cidade", "salario_bruto", "idade"]


In [16]:
df = spark.createDataFrame(dados_ficticios, colunas)
print("--- DataFrame Original ---")
df.show()


--- DataFrame Original ---
+----------------+--------------+-------------+-----+
|            nome|        cidade|salario_bruto|idade|
+----------------+--------------+-------------+-----+
|     Ana Silva  |     SÃO PAULO|  R$ 5.000,00| NULL|
|     joao santos|rio de janeiro|  R$ 3.200,50|   25|
| Maria Oliveira |Belo Horizonte| R$ 12.000,00|   40|
|    Carlos souza|          NULL|  R$ 1.500,00|   17|
+----------------+--------------+-------------+-----+



In [35]:
df_tratado = df.withColumn("nome", trim(lower(col("nome")))) \
.withColumn("cidade", coalesce(col("cidade"), lit("Não Informado"))) \
.withColumn("cidade", lower(col("cidade"))) \
.withColumn("salario_limpo", regexp_replace(col("salario_bruto"), r"[R$\s.]", "")) \
.withColumn("salario_limpo", regexp_replace(col("salario_limpo"), ",", ".").cast("float")) \
.withColumn("categoria_idade",
                when(col("idade").isNull(), "Não Identificado")
                .when(col("idade") >= 18, "Maior de Idade")
                .otherwise("Menor de Idade")).drop("salario_bruto")


In [24]:
print("--- DataFrame Tratado ---")
df_tratado.show()

--- DataFrame Tratado ---
+--------------+--------------+-----+-------------+----------------+
|          nome|        cidade|idade|salario_limpo| categoria_idade|
+--------------+--------------+-----+-------------+----------------+
|     ana silva|     são paulo| NULL|       5000.0|Não Identificado|
|   joao santos|rio de janeiro|   25|       3200.5|  Maior de Idade|
|maria oliveira|belo horizonte|   40|      12000.0|  Maior de Idade|
|  carlos souza| não informado|   17|       1500.0|  Menor de Idade|
+--------------+--------------+-----+-------------+----------------+



In [34]:
df_tratado.agg(sum("salario_limpo")).show()

+------------------+
|sum(salario_limpo)|
+------------------+
|           21700.5|
+------------------+

